# 01 - Exploratory Data Analysis

**Pin-To-Place: Location & Positional Accuracy**

This notebook explores the Overture Maps places dataset to understand:
- Geographic distribution of places
- Category breakdown
- Confidence score distribution
- Data source analysis
- Missing data patterns
- Near-duplicate detection

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_places, find_near_duplicates

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)

df = load_places()
print(f'Loaded {len(df)} places')
df.head(3)

## 1. Dataset Overview

In [ ]:
print('=== Shape ===')
print(f'{df.shape[0]} rows x {df.shape[1]} columns')
print()

print('=== Key Column Types ===')
key_cols = ['id', 'lon', 'lat', 'name', 'category_primary', 'confidence',
            'region', 'locality', 'freeform', 'source_count', 'full_address']
for col in key_cols:
    null_pct = df[col].isna().mean() * 100
    print(f'  {col:25s} nulls: {null_pct:.1f}%  dtype: {df[col].dtype}')

print()
print('=== Null Summary (original columns) ===')
orig_cols = ['websites', 'socials', 'emails', 'phones', 'brand']
for col in orig_cols:
    null_count = df[col].isna().sum()
    print(f'  {col:15s} nulls: {null_count:>5d} ({null_count/len(df)*100:.1f}%)')

## 2. Geographic Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot of all places
ax = axes[0]
ax.scatter(df['lon'], df['lat'], alpha=0.3, s=5, c='steelblue')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'All {len(df)} Places - Geographic Distribution')

# Top 15 states
ax = axes[1]
top_states = df['region'].value_counts().head(15)
top_states.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Top 15 States by Place Count')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

print(f'Lon range: {df["lon"].min():.4f} to {df["lon"].max():.4f}')
print(f'Lat range: {df["lat"].min():.4f} to {df["lat"].max():.4f}')
print(f'Unique states/regions: {df["region"].nunique()}')

## 3. Category Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 categories
ax = axes[0]
top_cats = df['category_primary'].value_counts().head(20)
top_cats.plot.barh(ax=ax, color='coral')
ax.set_xlabel('Count')
ax.set_title('Top 20 Primary Categories')
ax.invert_yaxis()

# Category group distribution
ax = axes[1]
from src.features import categorize_place
df['category_group'] = df['category_primary'].apply(categorize_place)
group_counts = df['category_group'].value_counts()
group_counts.plot.pie(ax=ax, autopct='%1.1f%%', startangle=90)
ax.set_ylabel('')
ax.set_title('Category Groups')

plt.tight_layout()
plt.show()

print(f'Unique primary categories: {df["category_primary"].nunique()}')
print(f'\nCategory group breakdown:')
print(group_counts)

## 4. Confidence Score Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
df['confidence'].hist(bins=50, ax=ax, color='teal', alpha=0.7)
ax.axvline(df['confidence'].median(), color='red', linestyle='--', label=f'Median: {df["confidence"].median():.3f}')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Count')
ax.set_title('Confidence Score Distribution')
ax.legend()

# Confidence by category group
ax = axes[1]
df.boxplot(column='confidence', by='category_group', ax=ax, rot=45)
ax.set_title('Confidence by Category Group')
ax.set_xlabel('Category Group')
ax.set_ylabel('Confidence')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(df['confidence'].describe())

## 5. Data Source Analysis

In [ ]:
# Source dataset frequency
all_sources = []
for sources in df['source_datasets']:
    all_sources.extend(sources)

source_series = pd.Series(all_sources)
print('=== Source Dataset Distribution ===')
print(source_series.value_counts())
print()

# Source count distribution
fig, ax = plt.subplots(figsize=(8, 4))
df['source_count'].value_counts().sort_index().plot.bar(ax=ax, color='steelblue')
ax.set_xlabel('Number of Sources')
ax.set_ylabel('Count')
ax.set_title('Number of Data Sources per Place')
plt.tight_layout()
plt.show()

print(f'\nSource count stats:')
print(df['source_count'].describe())

## 6. Address Quality

In [ ]:
print('=== Address Completeness ===')
addr_cols = ['freeform', 'locality', 'region', 'postcode', 'country']
for col in addr_cols:
    non_null = df[col].notna().sum()
    print(f'  {col:12s}: {non_null:>5d} ({non_null/len(df)*100:.1f}%)')

# Places with full addresses (all fields present)
has_full_addr = df[addr_cols].notna().all(axis=1).sum()
print(f'\n  Full address: {has_full_addr:>5d} ({has_full_addr/len(df)*100:.1f}%)')

# Sample addresses
print('\n=== Sample Full Addresses ===')
for addr in df['full_address'].sample(5, random_state=42):
    print(f'  {addr}')

## 7. Near-Duplicate Detection

In [ ]:
dupes = find_near_duplicates(df, max_distance_m=50.0)
print(f'Near-duplicates (same name within 50m): {len(dupes)}')
if len(dupes) > 0:
    print(dupes.head(20))
else:
    print('No near-duplicates found - dataset is clean.')

# Also check for same address, different coords
addr_groups = df.groupby('full_address').filter(lambda x: len(x) > 1)
print(f'\nPlaces sharing exact same address: {len(addr_groups)}')
if len(addr_groups) > 0:
    print(addr_groups[['name', 'category_primary', 'full_address', 'lat', 'lon']].head(10))

## 8. Summary Statistics for Stratified Sampling

In [ ]:
print('=== Stratification Preview ===')
print(f'\nTotal places: {len(df)}')
print(f'Unique regions: {df["region"].nunique()}')
print(f'Unique categories: {df["category_primary"].nunique()}')

# Top 10 region x category group combinations
strata = df.groupby(['region', 'category_group']).size().reset_index(name='count')
strata = strata.sort_values('count', ascending=False)
print(f'\nTotal strata (region x category_group): {len(strata)}')
print(f'\nTop 10 strata:')
print(strata.head(10).to_string(index=False))

# Target sample of 750
from src.ground_truth import stratified_sample
sample = stratified_sample(df, n=750)
print(f'\n=== Stratified Sample (n=750) ===')
print(f'Sampled: {len(sample)}')
print(f'Regions covered: {sample["region"].nunique()}')
print(f'Categories covered: {sample["category_primary"].nunique()}')